In [6]:

#SIGNALS CODE

#Installing Python Libraries
import pandas as pd
import os

#Read Data
df = pd.read_csv("clean_data.csv")

#Returns
df['1d_return'] = df['close'].pct_change()
df['5d_return'] = df['close'].pct_change(5)
df['20d_return'] = df['close'].pct_change(20)

#Distance From 10-Day SMA
df['10ma'] = df['close'].rolling(window=10).mean()
df['percent_ma_gap'] = (df['close'] - df['10ma']) / df['10ma'] * 100

#10-Day/ 50-Day SMA Crossover
df['50ma'] = df['close'].rolling(window=50).mean()
df['sma_crossover'] = (df['10ma'] > df['50ma']).astype(int)

#20-Day Drawdown
df['20d_drawdown'] = df['close'] / df['close'].rolling(window=20).max() - 1

#10-Day Realized Volatility
df['10d_volatility'] = df['1d_return'].rolling(window=10).std()

#Volatility Ratio
df['20d_volatility'] = df['1d_return'].rolling(window=20).std()

df['volatility_ratio'] = df['10d_volatility'] / df['20d_volatility']

#20-Day Volume z-score
df['volume_zscore'] = (
    (df['volume'] - df['volume'].rolling(window=20).mean()) /
    df['volume'].rolling(window=20).std()
)

#14 day RSI
delta = df['close'].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss

df['rsi_14'] = 100 - (100 / (1 + rs))

#The code below will show the outcome and is not part of the signals used in the model
#Future 5-Day Returndf['future_5d_return'] = df['close'].shift(-5) / df['close'] - 1
#Binary Target (1 if positive, 0 if otherwise)df['target'] = (df['future_5d_return'] > 0).astype(int)
#save to csv
os.makedirs('data', exist_ok=True)
df.to_csv("data/signals.csv", index=False)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')